# Callback inventory
> Where each pipeline callback is defined, and which names are defined in more than one place

A handler builds its pipeline from shared callbacks in `marisco.callbacks` and `marisco.metadata`, and from callbacks it defines itself. A handler callback can have the same name as a shared callback or as a callback in another handler. Same-named callbacks can do different things, and identical code can be copied between handlers without anyone noticing.

This page lists every callback and the module that defines it, and it shows which names are defined more than once. The tables are generated from the installed modules: run the notebook again to update them.

In [ ]:
import importlib, inspect, pkgutil, pandas as pd
import marisco.callbacks, marisco.metadata, marisco.handlers
from marisco.callbacks import Callback

In [ ]:
#| hide
pd.set_option('display.max_rows', None, 'display.max_colwidth', None)

## Modules scanned

The scan covers the two shared modules and every module in `marisco.handlers`. A handler added later is included without a change to this page.

In [ ]:
mods = {m.__name__.removeprefix('marisco.'): m for m in [marisco.callbacks, marisco.metadata] + [
    importlib.import_module(f'marisco.handlers.{i.name}') for i in pkgutil.iter_modules(marisco.handlers.__path__)]}
list(mods)

['callbacks',
 'metadata',
 'handlers.fram_strait',
 'handlers.geotraces',
 'handlers.helcom',
 'handlers.jois',
 'handlers.maris_legacy',
 'handlers.ospar',
 'handlers.tepco']

A callback belongs to the module that defines it. Names that a module only imports are skipped, so a handler that imports a shared callback doesn't list it again:

In [ ]:
def module_cbs(m):
    "Callback classes defined in module `m`, by name"
    return {k:v for k,v in vars(m).items()
        if inspect.isclass(v) and issubclass(v, Callback) and v is not Callback and v.__module__==m.__name__}

In [ ]:
cbs = pd.DataFrame([
    dict(module=n, callback=k, base=v.__bases__[0].__name__, doc=inspect.cleandoc(v.__doc__ or '').split('\n')[0], cls=v)
    for n,m in mods.items() for k,v in module_cbs(m).items()])

## Callbacks per module

`handlers.tepco` exports no callbacks yet: its notebook defines them in cells that aren't exported.

In [ ]:
cbs.module.value_counts(sort=False).reindex(list(mods), fill_value=0)

module
callbacks                13
metadata                  6
handlers.fram_strait      7
handlers.geotraces       11
handlers.helcom          17
handlers.jois             7
handlers.maris_legacy     3
handlers.ospar           11
handlers.tepco            0
Name: count, dtype: int64

## All callbacks

Each row gives the base class and the first line of the docstring. The docstring is also the text a callback writes to the NetCDF processing log.

In [ ]:
cbs.drop(columns='cls').set_index(['module', 'callback'])

base                                                                                                                                                                                                                    doc
module                callback                                                                                                                                                                                                                                                   
callbacks             PerGroupCB                  Callback                                                                                                                                Calls `each_grp` for each group in `tfm.dfs`; set `grps` to restrict to specific groups
                      SanitizeLonLatCB          PerGroupCB                                                                                                                             Drop rows with invalid longitude & latitude values. Convert `,` separator to `.` separator
                      RemoveAllNAValuesCB         Callback                                                                                                                                                                    Remove rows with all NA values in specified columns
                      RemapCB                   PerGroupCB                                                                                                                                                 Remap source values to MARIS standard identifiers using a lookup table
                      LowerStripNameCB          PerGroupCB                                                                                                                                                              Convert values to lowercase and strip any trailing spaces
                      AddSampleTypeIdColumnCB   PerGroupCB                                                                                                                                                                Add a column with the sample type as defined in the CDL
                      RenameColumnsCB           PerGroupCB                                                                                                                                                 Rename variables to MARIS standard names, keeping only renamed columns
                      MeltWideNuclidesCB          Callback                                                                                                                                                    Reshape wide nuclide columns to long format using a named-dict spec
                      AddSampleIDCB             PerGroupCB                                                                                                            Assign 1-based sequential SMP_ID; optionally cast a provider ID column to str for NetCDF VLEN compatibility
                      UniqueIndexCB             PerGroupCB                                                                                                                                                                                        Set unique index for each group
                      ParseTimeCB               PerGroupCB                                                                                                                                                                      Parse time column from ISO8601 string to datetime
                      EncodeTimeCB              PerGroupCB                                                                                                                                                                                     Encode time as seconds since epoch
                      DecodeTimeCB              PerGroupCB                                                                                                                                                                

## Names defined more than once

`implementations` counts the different source texts among the definitions of a name. A count of 1 means every definition is an exact copy, so the callback can move to `marisco.callbacks` as it is. A higher count means the definitions differ, and someone has to compare them before they can be shared or renamed (see [#82](https://github.com/franckalbinet/marisco/issues/82)).

In [ ]:
dups = cbs[cbs.callback.duplicated(keep=False)].groupby('callback').agg(
    modules=('module', ', '.join), implementations=('cls', lambda s: s.map(inspect.getsource).nunique()))
dups

,modules,implementations
callback,,
AddDepthCB,"handlers.fram_strait, handlers.helcom, handlers.ospar",3
AddDetectionLimitCB,"handlers.fram_strait, handlers.jois",1
AddSampleIDCB,"callbacks, handlers.geotraces, handlers.helcom, handlers.maris_legacy, handlers.ospar",5
AddStationCB,"handlers.helcom, handlers.ospar",2
ConvertU238CB,"handlers.fram_strait, handlers.jois",1
NormalizeUncCB,"handlers.helcom, handlers.ospar",2
ParseCoordinatesCB,"handlers.helcom, handlers.ospar",2
ParseDateTimeCB,"handlers.fram_strait, handlers.jois",2
ParseTimeCB,"callbacks, handlers.helcom, handlers.ospar",3
